## 1. Подготовка окружения

Ставим Ultralytics, пишем YAML-конфиги датасета и кастомной
архитектуры, скачиваем датасет.

In [ ]:
# Установка Ultralytics.
!pip install -q ultralytics

In [ ]:
import torch
import ultralytics

print("ultralytics:", ultralytics.__version__)
print("torch      :", torch.__version__)
print("cuda       :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device     :", torch.cuda.get_device_name(0))

In [ ]:
# Создаём папку configs/, чтобы держать YAML-файлы в одном месте.
import os
os.makedirs("configs", exist_ok=True)

In [ ]:
%%writefile configs/dataset.yaml
# Конфигурация датасета African Wildlife.

path: african-wildlife
train: images/train
val: images/val
test: images/test

nc: 4
names:
  0: buffalo
  1: elephant
  2: rhino
  3: zebra

download: https://github.com/ultralytics/assets/releases/download/v0.0.0/african-wildlife.zip

In [ ]:
%%writefile configs/custom_detector_t.yaml
# Кастомная архитектура детектора.
# За основу взята структура YOLOv8 (FPN+PAN, три выходные головы),
# но в собственном "tiny"-варианте: уменьшены depth, width, max_channels —
# модель легче yolo11n. Pretrained веса не используются (обучение с нуля).

nc: 4

scales:
  # depth_multiple, width_multiple, max_channels
  t: [0.25, 0.20, 512]

backbone:
  - [-1, 1, Conv,   [64, 3, 2]]    # 0-P1/2
  - [-1, 1, Conv,   [128, 3, 2]]   # 1-P2/4
  - [-1, 3, C2f,    [128, True]]   # 2
  - [-1, 1, Conv,   [256, 3, 2]]   # 3-P3/8
  - [-1, 6, C2f,    [256, True]]   # 4
  - [-1, 1, Conv,   [512, 3, 2]]   # 5-P4/16
  - [-1, 6, C2f,    [512, True]]   # 6
  - [-1, 1, Conv,   [1024, 3, 2]]  # 7-P5/32
  - [-1, 3, C2f,    [1024, True]]  # 8
  - [-1, 1, SPPF,   [1024, 5]]     # 9

head:
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 6], 1, Concat, [1]]
  - [-1, 3, C2f, [512]]                          # 12

  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 4], 1, Concat, [1]]
  - [-1, 3, C2f, [256]]                          # 15 (P3/8-small)

  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 12], 1, Concat, [1]]
  - [-1, 3, C2f, [512]]                          # 18 (P4/16-medium)

  - [-1, 1, Conv, [512, 3, 2]]
  - [[-1, 9], 1, Concat, [1]]
  - [-1, 3, C2f, [1024]]                         # 21 (P5/32-large)

  - [[15, 18, 21], 1, Detect, [nc]]

In [ ]:
# Скачиваем датасет штатным механизмом Ultralytics.

from ultralytics.data.utils import check_det_dataset

info = check_det_dataset("configs/dataset.yaml")
print("dataset path:", info["path"])
print("classes     :", info["names"])

In [ ]:
# Беглый sanity-check: считаем картинки и разметки в каждом сплите.
from pathlib import Path

dataset_dir = Path(info["path"])
for split in ("train", "val", "test"):
    images = list((dataset_dir / "images" / split).glob("*"))
    labels = list((dataset_dir / "labels" / split).glob("*.txt"))
    print(f"  {split:5}: {len(images):4d} изображений, {len(labels):4d} разметок")

In [ ]:
# Общие утилиты, которые используются во всех экспериментах.
import json
from pathlib import Path

RUNS_DIR = Path("runs")
RUNS_DIR.mkdir(exist_ok=True)


def _to_float(x):
    try:
        return float(x) if x is not None else None
    except (TypeError, ValueError):
        return None


def extract_metrics(results):
    '''Достаём Precision/Recall/mAP@0.5/mAP@0.5:0.95 из объекта Ultralytics.'''
    d = getattr(results, "results_dict", None) or {}
    m = {
        "precision": _to_float(d.get("metrics/precision(B)")),
        "recall":    _to_float(d.get("metrics/recall(B)")),
        "map50":     _to_float(d.get("metrics/mAP50(B)")),
        "map50_95":  _to_float(d.get("metrics/mAP50-95(B)")),
    }
    # Запасной путь — через box-метрики.
    if m["map50"] is None and hasattr(results, "box"):
        box = results.box
        m["precision"] = m["precision"] or _to_float(getattr(box, "mp", None))
        m["recall"]    = m["recall"]    or _to_float(getattr(box, "mr", None))
        m["map50"]     = m["map50"]     or _to_float(getattr(box, "map50", None))
        m["map50_95"] = m["map50_95"] or _to_float(getattr(box, "map", None))
    return m


def save_metrics(name, metrics):
    out = RUNS_DIR / name / "metrics.json"
    out.parent.mkdir(parents=True, exist_ok=True)
    with out.open("w", encoding="utf-8") as f:
        json.dump(metrics, f, ensure_ascii=False, indent=2)
    return out


def print_metrics(title, metrics):
    print(f"\n=== {title} ===")
    for k in ("precision", "recall", "map50", "map50_95"):
        v = metrics.get(k)
        print(f"  {k:10s}: {'n/a' if v is None else f'{v:.4f}'}")

## 2. Baseline — YOLO11n + pretrained

Базовая точка отсчёта. Берём самую лёгкую модель семейства YOLOv11
(`yolo11n`, ~2.6M параметров) с предобученными на COCO весами,
стандартные аугментации Ultralytics, 10 эпох.

In [ ]:
from ultralytics import YOLO

baseline_model = YOLO("yolo11n.pt")  # pretrained веса COCO

baseline_train = baseline_model.train(
    data="configs/dataset.yaml",
    epochs=10,
    batch=16,
    imgsz=640,
    device=0,                # GPU 0
    project="runs",
    name="baseline",
    exist_ok=True,
    # Стандартные аугментации, заданы явно для прозрачности отчёта.
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    degrees=0.0, translate=0.1, scale=0.5,
    fliplr=0.5, mosaic=1.0, mixup=0.0,
)

In [ ]:
# Финальная валидация baseline и сохранение метрик.
baseline_val = baseline_model.val(
    data="configs/dataset.yaml",
    imgsz=640,
    device=0,
    project="runs",
    name="baseline_val",
    exist_ok=True,
)

baseline_metrics = extract_metrics(baseline_val)
baseline_metrics.update({
    "experiment": "baseline",
    "model": "yolo11n.pt",
    "pretrained": True,
    "epochs": 10,
    "augmentations": "default",
})
save_metrics("baseline", baseline_metrics)
print_metrics("Baseline (YOLO11n)", baseline_metrics)

## 3. Improved — YOLO11s + усиленные аугментации

**Гипотезы:**

1. Более крупная модель `YOLO11s` (~9.4M параметров) лучше извлекает признаки
   на сложных кадрах с дальним планом, перекрытиями и плотным фоном.
2. Усиленные аугментации (`mixup`, `copy_paste`, расширенные HSV/scale/degrees,
   небольшая перспектива) расширяют эффективное распределение обучающих
   примеров и улучшают обобщающую способность.
3. 15 эпох вместо 10 + cosine LR-расписание — больше времени на сходимость.

In [ ]:
improved_model = YOLO("yolo11s.pt")  # более крупная архитектура, pretrained

improved_train = improved_model.train(
    data="configs/dataset.yaml",
    epochs=15,
    batch=16,
    imgsz=640,
    device=0,
    project="runs",
    name="improved",
    exist_ok=True,
    # Усиленные аугментации.
    hsv_h=0.02, hsv_s=0.8, hsv_v=0.5,
    degrees=10.0, translate=0.15, scale=0.6,
    shear=2.0, perspective=0.0005,
    fliplr=0.5, mosaic=1.0,
    mixup=0.15, copy_paste=0.1,
    # Cosine LR — плавное снижение learning rate к концу обучения.
    lr0=0.01, cos_lr=True,
)

In [ ]:
improved_val = improved_model.val(
    data="configs/dataset.yaml",
    imgsz=640,
    device=0,
    project="runs",
    name="improved_val",
    exist_ok=True,
)

improved_metrics = extract_metrics(improved_val)
improved_metrics.update({
    "experiment": "improved",
    "model": "yolo11s.pt",
    "pretrained": True,
    "epochs": 15,
    "augmentations": "strong (mixup + copy_paste)",
})
save_metrics("improved", improved_metrics)
print_metrics("Improved (YOLO11s)", improved_metrics)

## 4. Кастомная архитектура — режим default

Архитектура описана в `configs/custom_detector_t.yaml` — это
урезанный вариант YOLOv8 (depth=0.25, width=0.20, max_channels=512),
обучается **с нуля без pretrained-весов**. Аугментации — те же,
что в baseline (default).

Это даёт чистое сравнение «pretrained YOLO11n vs обучаемая с нуля
кастомная архитектура».

In [ ]:
# YOLO("<.yaml>") создаёт модель со случайными весами по описанию архитектуры.
custom_default_model = YOLO("configs/custom_detector_t.yaml")

custom_default_train = custom_default_model.train(
    data="configs/dataset.yaml",
    epochs=15,
    batch=16,
    imgsz=640,
    device=0,
    project="runs",
    name="custom_default",
    exist_ok=True,
    pretrained=False,
    # Дефолтные аугментации (как в baseline).
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    degrees=0.0, translate=0.1, scale=0.5,
    fliplr=0.5, mosaic=1.0, mixup=0.0,
)

In [ ]:
custom_default_val = custom_default_model.val(
    data="configs/dataset.yaml",
    imgsz=640,
    device=0,
    project="runs",
    name="custom_default_val",
    exist_ok=True,
)

custom_default_metrics = extract_metrics(custom_default_val)
custom_default_metrics.update({
    "experiment": "custom_default",
    "model": "custom_detector_t.yaml",
    "pretrained": False,
    "epochs": 15,
    "augmentations": "default",
})
save_metrics("custom_default", custom_default_metrics)
print_metrics("Custom default", custom_default_metrics)

## 5. Кастомная архитектура — режим improved

Та же архитектура, что и в предыдущем эксперименте, но с усиленными
аугментациями (как в improved). Это позволяет внутри кастомной модели
проверить, насколько аугментации компенсируют отсутствие pretrained-весов.

In [ ]:
custom_improved_model = YOLO("configs/custom_detector_t.yaml")

custom_improved_train = custom_improved_model.train(
    data="configs/dataset.yaml",
    epochs=20,
    batch=16,
    imgsz=640,
    device=0,
    project="runs",
    name="custom_improved",
    exist_ok=True,
    pretrained=False,
    # Усиленные аугментации.
    hsv_h=0.02, hsv_s=0.8, hsv_v=0.5,
    degrees=10.0, translate=0.15, scale=0.6,
    shear=2.0, perspective=0.0005,
    fliplr=0.5, mosaic=1.0,
    mixup=0.15, copy_paste=0.1,
    lr0=0.01, cos_lr=True,
)

In [ ]:
custom_improved_val = custom_improved_model.val(
    data="configs/dataset.yaml",
    imgsz=640,
    device=0,
    project="runs",
    name="custom_improved_val",
    exist_ok=True,
)

custom_improved_metrics = extract_metrics(custom_improved_val)
custom_improved_metrics.update({
    "experiment": "custom_improved",
    "model": "custom_detector_t.yaml",
    "pretrained": False,
    "epochs": 20,
    "augmentations": "strong",
})
save_metrics("custom_improved", custom_improved_metrics)
print_metrics("Custom improved", custom_improved_metrics)

## 6. Итоговое сравнение

Сводим все четыре эксперимента в одну таблицу и строим столбчатую диаграмму
по основным метрикам.

In [ ]:
import pandas as pd

rows = []
for name in ("baseline", "improved", "custom_default", "custom_improved"):
    path = RUNS_DIR / name / "metrics.json"
    if not path.exists():
        print(f"[warn] нет метрик для {name}")
        continue
    with path.open(encoding="utf-8") as f:
        data = json.load(f)
    rows.append({
        "experiment": name,
        "model":      data.get("model"),
        "epochs":     data.get("epochs"),
        "pretrained": data.get("pretrained"),
        "augs":       data.get("augmentations"),
        "precision":  data.get("precision"),
        "recall":     data.get("recall"),
        "mAP@0.5":    data.get("map50"),
        "mAP@0.5:0.95": data.get("map50_95"),
    })

df = pd.DataFrame(rows).set_index("experiment")
df_round = df.copy()
for col in ("precision", "recall", "mAP@0.5", "mAP@0.5:0.95"):
    df_round[col] = df_round[col].astype(float).round(4)
df_round

In [ ]:
# Бар-чарт по mAP@0.5 и mAP@0.5:0.95
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
metric_cols = ["precision", "recall", "mAP@0.5", "mAP@0.5:0.95"]
df[metric_cols].astype(float).plot.bar(ax=ax)
ax.set_ylabel("значение метрики")
ax.set_ylim(0, 1)
ax.set_title("Сравнение экспериментов")
ax.legend(loc="lower right", fontsize=8)
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()